In [ ]:
import subprocess, sys, os, importlib

# -- 0. Clone / pull repo ------------------------------------
REPO_URL = 'https://github.com/bishnt/PINN_for_transient_stability.git'
REPO_DIR = '/content/pinn-transient-stability'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'torch', 'numpy', 'scipy', 'matplotlib', 'tqdm', 'scikit-learn'],
    check=True
)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.chdir(REPO_DIR)

# -- Print latest commits (useful to confirm stale-import issues) --
result = subprocess.run(
    ['git', 'log', '--oneline', '-3'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(f'[git] Latest commits:\n{result.stdout}')
print(f'[OK] Working directory: {os.getcwd()}')

# -- Force reload all project modules after pull --------------
for mod_name in ['swing_equation', 'data_generator', 'model', 'loss', 'trainer', 'stability_analysis']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from swing_equation import SMIBParameters
from data_generator import PINNDataGenerator
from model import PINN, count_parameters
from stability_analysis import StabilityAnalyzer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# -- 1. Load trained model ------------------------------------
MODEL_PATH = 'models/pinn_trained.pt'

if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model = PINN(n_hidden_layers=4, n_neurons=64)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f'[OK] Loaded trained model from {MODEL_PATH}')
    print(f'     Trainable parameters: {count_parameters(model):,}')
else:
    print(f'[ERROR] Model file not found: {MODEL_PATH}')
    print('Please train the model first using the training notebook.')
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')

In [ ]:
# -- 2. Setup parameters for stability analysis -------------
# Physics parameters (should match training)
params = SMIBParameters(H=5.0, D=0.05, Pm=0.8, Pmax=2.1)

# Fault parameters
FAULT_START = 0.1
FAULT_END = 0.2
T_TOTAL = 2.0

# Stability threshold (degrees)
DELTA_THRESHOLD = 120.0

# Initialize stability analyzer
analyzer = StabilityAnalyzer(model, params, device)
print(f'[OK] Stability analyzer initialized')
print(f'     Equilibrium angle: {np.degrees(params.delta_eq):.2f} degrees')

In [ ]:
# -- 3. Generate reference trajectory for accuracy check ----
gen = PINNDataGenerator(params, seed=42)
traj = gen.generate_reference_trajectory(
    fault_start=FAULT_START, 
    fault_end=FAULT_END, 
    t_total=T_TOTAL
)

print(f'[OK] Reference trajectory generated')
print(f'     Time span: 0 to {T_TOTAL}s')
print(f'     Fault window: {FAULT_START}s to {FAULT_END}s')

In [ ]:
# -- 4. Evaluate PINN accuracy against RK4 -----------------
results = analyzer.evaluate_accuracy(traj, n_points=500)

print('\n=== Accuracy Metrics ===')
print(f'MAE delta: {np.degrees(results["mae_delta"]):.4f} degrees')
print(f'MAE omega: {results["mae_omega"]:.4f} rad/s')
print(f'RMSE delta: {np.degrees(results["rmse_delta"]):.4f} degrees')
print(f'RMSE omega: {results["rmse_omega"]:.4f} rad/s')
print('========================\n')

In [ ]:
# -- 5. Plot PINN vs RK4 comparison ------------------------
os.makedirs('results/plots', exist_ok=True)
analyzer.plot_comparison(results, save_path='results/plots/stability_comparison.png')

In [ ]:
# -- 6. Compute stability map using RK4 ---------------------
# Define parameter sweep ranges
# Use wider range to ensure we find both stable and unstable regions
fault_durations = np.linspace(0.05, 0.8, 25)  # 50ms to 800ms
initial_angles = np.linspace(np.radians(5), np.radians(85), 35)  # 5 to 85 degrees

print(f'Computing stability map using RK4...')
print(f'  Fault durations: {len(fault_durations)} points ({fault_durations[0]*1000:.0f}ms to {fault_durations[-1]*1000:.0f}ms)')
print(f'  Initial angles: {len(initial_angles)} points ({np.degrees(initial_angles[0]):.0f}° to {np.degrees(initial_angles[-1]):.0f}°)')
print('  Note: PINN is trained on a single fault scenario and cannot generalize')
print('        to different fault durations or initial angles.')
print('        Using RK4 for stability analysis across parameter space.')

# Compute using RK4 (PINN not suitable for parameter sweeps)
stability_map = analyzer.compute_stability_map(
    fault_durations,
    initial_angles,
    t_total=T_TOTAL,
    delta_threshold_deg=DELTA_THRESHOLD,
    use_pinn=False  # Use RK4
)

print(f'[OK] Stability map computed')
print(f'     Stable regions: {stability_map.sum()} / {stability_map.size}')
print(f'     Percentage stable: {stability_map.sum()/stability_map.size*100:.1f}%')

# Check if we found any unstable regions
if stability_map.sum() == stability_map.size:
    print('  WARNING: All regions are stable! Try increasing fault duration range or threshold.')
else:
    print(f'  Found unstable regions: {stability_map.size - stability_map.sum()} / {stability_map.size}')

In [ ]:
# -- 7. Plot stability boundary map ---------------------------
analyzer.plot_stability_map(
    fault_durations,
    initial_angles,
    stability_map,
    save_path='results/plots/stability_map.png'
)

In [ ]:
# -- 8. Summary -----------------------------------------------
print('\n=== Stability Evaluation Summary ===')
print(f'PINN accuracy on training scenario:')
print(f'  - RMSE delta: {np.degrees(results["rmse_delta"]):.4f} degrees')
print(f'  - RMSE omega: {results["rmse_omega"]:.4f} rad/s')
print(f'\nStability boundary analysis:')
print(f'  - Parameter space: {len(fault_durations)} fault durations × {len(initial_angles)} initial angles')
print(f'  - Fault duration range: {fault_durations[0]*1000:.0f}ms to {fault_durations[-1]*1000:.0f}ms')
print(f'  - Initial angle range: {np.degrees(initial_angles[0]):.0f}° to {np.degrees(initial_angles[-1]):.0f}°')
print(f'  - Stable regions: {stability_map.sum()} / {stability_map.size} ({stability_map.sum()/stability_map.size*100:.1f}%)')

# Find approximate critical fault duration (where 50% of cases are unstable)
if stability_map.sum() < stability_map.size:
    stability_ratio = stability_map.mean(axis=0)
    critical_idx = np.where(stability_ratio < 0.5)[0]
    if len(critical_idx) > 0:
        critical_fault = fault_durations[critical_idx[0]]
        print(f'  - Critical fault duration: ~{critical_fault*1000:.0f}ms (50% unstable)')
    else:
        print(f'  - No clear critical fault duration found in range')
else:
    print(f'  - All tested scenarios are stable')

print('===================================\n')

print('[OK] Stability evaluation complete!')
print('Results saved to results/plots/')